<h3 style="color:#6FA8DC; font-weight:bold">Python Web Scraping — Complete Guide</h3>

This notebook covers Requests, BeautifulSoup, lxml, HTTPX, Playwright, Scrapy, pagination, cleaning and exporting.

We will scrape **Hacker News**, a public website that does not require an API key.

<h5 style="color:#78B89A; font-weight:bold;">1. What is Web Scraping?</h5>

Web scraping means automatically collecting information from webpages.

```text
Fetch webpage → Parse HTML → Select elements → Extract data → Clean → Store
```

- Scraping: extracting useful data.
- Crawling: discovering and visiting many pages.

<h5 style="color:#78B89A; font-weight:bold;">2. Tools and their uses</h5>

| Tool | Purpose |
|---|---|
| requests | Fetch HTML |
| httpx | Modern sync/async HTTP client |
| BeautifulSoup | Parse HTML |
| lxml | Fast parsing and XPath |
| Playwright | JavaScript-rendered pages |
| Scrapy | Large-scale crawling |
| Pandas | Clean and analyze data |

**Learning order:** Requests → BeautifulSoup → CSS selectors → Pagination → Pandas → lxml → HTTPX → Playwright → Scrapy.

<h5 style="color:#78B89A; font-weight:bold;">3. Installation</h5>

In [ ]:
# Run these in terminal or uncomment inside Jupyter
# %pip install requests beautifulsoup4 lxml pandas httpx
# %pip install playwright
# !playwright install chromium
# %pip install scrapy

<h5 style="color:#78B89A; font-weight:bold;">4. Import libraries</h5>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

<h5 style="color:#78B89A; font-weight:bold;">5. Fetch a webpage using Requests</h5>

In [ ]:
url = "https://news.ycombinator.com/"

response = requests.get(url, timeout=15)

response.status_code

In [ ]:
response.raise_for_status()

html = response.text

print(html[:1000])

<h5 style="color:#78B89A; font-weight:bold;">6. Parse HTML using BeautifulSoup</h5>

In [ ]:
soup = BeautifulSoup(html, "html.parser")

print(soup.title.get_text(strip=True))

In [ ]:
# lxml is another parser backend
soup_lxml = BeautifulSoup(html, "lxml")

print(soup_lxml.title.get_text(strip=True))

<h5 style="color:#78B89A; font-weight:bold;">7. Important BeautifulSoup methods</h5>

| Method | Use |
|---|---|
| `find()` | First matching element |
| `find_all()` | All matching elements |
| `select_one()` | First CSS selector match |
| `select()` | All CSS selector matches |
| `get_text()` | Extract text |
| `get()` | Extract attribute |
| `find_parent()` | Find parent |
| `find_next_sibling()` | Find next sibling |

In [ ]:
first_link = soup.find("a")

if first_link:
    print(first_link.get_text(strip=True))
    print(first_link.get("href"))

In [ ]:
all_links = soup.find_all("a")

len(all_links)

In [ ]:
for link in all_links[:10]:
    print(link.get_text(" ", strip=True))

<h5 style="color:#78B89A; font-weight:bold;">8. CSS selectors</h5>

| Selector | Meaning |
|---|---|
| `a` | All anchor tags |
| `.class` | Class selector |
| `#id` | ID selector |
| `div a` | Anchor inside div |
| `div > a` | Direct child anchor |
| `h1, h2` | h1 or h2 |

In [ ]:
soup.select("a")[:3]

In [ ]:
soup.select_one("title")

<h5 style="color:#78B89A; font-weight:bold;">9. Scrape Hacker News story titles and links</h5>

In [ ]:
stories = []

for story in soup.select("tr.athing"):
    title_element = story.select_one(".titleline > a")

    if title_element:
        stories.append({
            "title": title_element.get_text(" ", strip=True),
            "url": title_element.get("href", "")
        })

stories[:3]

In [ ]:
stories_df = pd.DataFrame(stories)

stories_df.head()

In [ ]:
stories_df.shape


<h5 style="color:#78B89A; font-weight:bold;">10. Extract story scores</h5>

In [ ]:
records = []

for story in soup.select("tr.athing"):
    title_element = story.select_one(".titleline > a")
    score_row = story.find_next_sibling("tr")
    score_element = score_row.select_one(".score") if score_row else None

    if title_element:
        records.append({
            "title": title_element.get_text(" ", strip=True),
            "url": title_element.get("href", ""),
            "score": score_element.get_text(strip=True) if score_element else "0 points"
        })

scores_df = pd.DataFrame(records)
scores_df.head()

In [ ]:
scores_df["score"] = (
    scores_df["score"]
    .str.replace(" points", "", regex=False)
    .astype(int)
)

scores_df.sort_values("score", ascending=False).head(10)

<h5 style="color:#78B89A; font-weight:bold;">11. Request headers</h5>

In [ ]:
headers = {
    "User-Agent": "EducationalDataCollector/1.0",
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(url, headers=headers, timeout=15)

response.status_code

<h5 style="color:#78B89A; font-weight:bold;">12. Error handling</h5>

In [ ]:
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    print("Page fetched successfully")

except requests.exceptions.Timeout:
    print("Request timed out")

except requests.exceptions.RequestException as error:
    print("Request failed:", error)

<h5 style="color:#78B89A; font-weight:bold;">13. Sessions</h5>

In [ ]:
session = requests.Session()
session.headers.update(headers)

response = session.get(url, timeout=15)
response.status_code

<h5 style="color:#78B89A; font-weight:bold;">14. Pagination</h5>

Hacker News supports the `p` query parameter. We will scrape three pages.

In [ ]:
all_stories = []

for page_number in range(1, 4):
    page_url = f"https://news.ycombinator.com/?p={page_number}"

    response = requests.get(
        page_url,
        headers=headers,
        timeout=15
    )
    response.raise_for_status()

    page_soup = BeautifulSoup(response.text, "lxml")

    for story in page_soup.select("tr.athing"):
        title_element = story.select_one(".titleline > a")

        if title_element:
            all_stories.append({
                "page": page_number,
                "title": title_element.get_text(" ", strip=True),
                "url": title_element.get("href", "")
            })

    time.sleep(1)

multi_page_df = pd.DataFrame(all_stories).drop_duplicates()
multi_page_df.head()

<h5 style="color:#78B89A; font-weight:bold;">15. Extract all links</h5>

In [ ]:
links_data = []

for link in soup.find_all("a"):
    links_data.append({
        "text": link.get_text(" ", strip=True),
        "href": link.get("href")
    })

links_df = pd.DataFrame(links_data)
links_df.head()

<h5 style="color:#78B89A; font-weight:bold;">16. Read HTML tables with Pandas</h5>

If a webpage contains standard HTML tables, Pandas can sometimes read them directly.

In [ ]:
# Example syntax:
# tables = pd.read_html("https://example.com/table-page")
# tables[0].head()

<h5 style="color:#78B89A; font-weight:bold;">17. XPath with lxml</h5>

In [ ]:
from lxml import html

tree = html.fromstring(response.text)

titles = tree.xpath(
    "//tr[contains(@class, 'athing')]//span[@class='titleline']/a/text()"
)

titles[:5]

<h5 style="color:#78B89A; font-weight:bold;">18. Modern HTTP client — HTTPX</h5>

HTTPX supports synchronous and asynchronous HTTP requests.

In [ ]:
# %pip install httpx

import httpx

with httpx.Client(timeout=15) as client:
    httpx_response = client.get(url)

httpx_response.status_code

In [ ]:
httpx_soup = BeautifulSoup(httpx_response.text, "lxml")
httpx_soup.title.get_text(strip=True)

<h5 style="color:#78B89A; font-weight:bold;">19. Playwright for JavaScript websites</h5>

BeautifulSoup cannot execute JavaScript. Playwright controls a real browser and can access content rendered after page load.

In [ ]:
# Install first:
# %pip install playwright
# !playwright install chromium

# Example:
# from playwright.sync_api import sync_playwright
#
# with sync_playwright() as p:
#     browser = p.chromium.launch(headless=True)
#     page = browser.new_page()
#     page.goto("https://example.com")
#     page.wait_for_selector(".product-card")
#     rendered_html = page.content()
#     browser.close()

<h5 style="color:#78B89A; font-weight:bold;">20. Common Playwright methods</h5>

| Method | Use |
|---|---|
| `page.goto()` | Open URL |
| `page.locator()` | Find element |
| `locator.click()` | Click |
| `locator.fill()` | Enter text |
| `locator.text_content()` | Extract text |
| `page.wait_for_selector()` | Wait for element |
| `page.content()` | Get rendered HTML |
| `browser.close()` | Close browser |

<h5 style="color:#78B89A; font-weight:bold;">21. Scrapy overview</h5>

Scrapy is useful for large-scale crawling. It provides spiders, scheduling, duplicate filtering, middleware, item pipelines and exports.

```bash
pip install scrapy
scrapy startproject news_project
scrapy crawl myspider
```

<h5 style="color:#78B89A; font-weight:bold;">22. Clean scraped data using Pandas</h5>

In [ ]:
clean_df = multi_page_df.drop_duplicates()
clean_df = clean_df.dropna(subset=["title"])
clean_df = clean_df.reset_index(drop=True)

clean_df.head()

<h5 style="color:#78B89A; font-weight:bold;">23. Save scraped data</h5>

In [ ]:
clean_df.to_csv("hacker_news_stories.csv", index=False)

clean_df.to_json(
    "hacker_news_stories.json",
    orient="records",
    indent=4
)

<h5 style="color:#78B89A; font-weight:bold;">24. Reusable scraper function</h5>

In [ ]:
def scrape_hacker_news(page_number=1):
    page_url = f"https://news.ycombinator.com/?p={page_number}"

    response = requests.get(
        page_url,
        headers=headers,
        timeout=15
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")
    records = []

    for story in soup.select("tr.athing"):
        title_element = story.select_one(".titleline > a")

        if title_element:
            records.append({
                "page": page_number,
                "title": title_element.get_text(" ", strip=True),
                "url": title_element.get("href", "")
            })

    return pd.DataFrame(records)

scrape_hacker_news(1).head()

<h5 style="color:#78B89A; font-weight:bold;">25. Responsible scraping</h5>

1. Check `robots.txt`.
2. Read Terms of Service.
3. Prefer official APIs when available.
4. Do not bypass authentication or access controls.
5. Avoid collecting private or sensitive data.
6. Add delays between requests.
7. Use timeouts and error handling.
8. Do not overload servers.
9. Store only required data.
10. Respect copyright and data-use restrictions.

<h3 style="color:#6FA8DC; font-weight:bold">Final Revision</h3>

```text
FETCH → PARSE → SELECT → EXTRACT → CLEAN → STORE → ANALYZE
```

| Situation | Tool |
|---|---|
| Static HTML | Requests + BeautifulSoup |
| XPath / fast parsing | lxml |
| Modern HTTP / async | HTTPX |
| JavaScript content | Playwright |
| Large crawling project | Scrapy |
| Cleaning and analysis | Pandas |